# SFT Model Verification — HF → Eval → F1 Check

Loads `OliverSlivka/qwen2.5-7b-itemset-extractor` from HuggingFace,
runs inference on N evaluation datasets, and prints F1 / parse / hallucination results.

**Expected:** avg F1 ≈ 0.13 (Iteration 4 SFT, primary_v3)

**Instructions:** Run Cells 1-7 in order. Cell 3 loads the model (slow, one-time).

In [ ]:
# ── Cell 1: Imports + Config ──────────────────────────────────────────────
from __future__ import annotations
import itertools, json, os, random, re, sys, time
from pathlib import Path

# ── Config ─────────────────────────────────────────────────────────────────
MODEL_REPO      = "OliverSlivka/qwen2.5-7b-itemset-extractor"
DATA_DIR        = "datasets_v2"   # relative to notebook working dir
EVAL_COUNT      = 10             # how many datasets to evaluate
EXCLUDE_TRAIN   = True           # skip training datasets
MIN_SUPPORT     = 3
MAX_SIZE        = 3
MAX_NEW_TOKENS  = 512
TEMPERATURE     = 0.3
TWO_PHASE       = False
SEED            = 42
OUTPUT_DIR      = "eval_verify_results"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
SCRIPT_DIR = Path.cwd()

# ── System prompt (must match training EXACTLY) ────────────────────────────
SYSTEM_PROMPT = (
    "You are a frequent itemset extractor. Given CSV transaction data and a "
    "minimum support count, identify all itemsets whose items co-occur in at "
    "least that many rows.\n\n"
    "Rules:\n"
    "1. Scan single items, pairs, and triples (up to size 3)\n"
    "2. Count = number of distinct rows containing ALL items in the itemset\n"
    "3. Only report itemsets with count >= min_support\n"
    "4. Canonicalize items: lowercase, trimmed, sorted alphabetically\n"
    '5. Row references: "Row N" format, 1-based indexing\n\n'
    "Think step by step inside <think> tags, then output ONLY a JSON array:\n"
    '[{"itemset": ["item1", "item2"], "count": N, "rows": ["Row 1", "Row 3"]}]'
)

print("Config loaded")
print(f"   Model: {MODEL_REPO}")
print(f"   Data:  {DATA_DIR}")
print(f"   Count: {EVAL_COUNT}  |  Exclude train: {EXCLUDE_TRAIN}")
print(f"   Temp:  {TEMPERATURE}  |  Two-phase: {TWO_PHASE}")


In [ ]:
# ── Cell 2: CSV loader + Apriori ground truth ─────────────────────────────

def load_csv(path: str):
    import pandas as pd
    df = pd.read_csv(path)
    transactions = []
    lines = []
    for idx, row in df.iterrows():
        items = [str(v).strip() for v in row.values if pd.notna(v) and str(v).strip()]
        transactions.append(items)
        lines.append(f"Row {idx+1}: {', '.join(items)}")
    return transactions, "\n".join(lines), len(df), len(df.columns)

def apriori(transactions, min_support=3, max_size=3):
    if not transactions:
        return []
    row_labels = [f"Row {i+1}" for i in range(len(transactions))]
    counts = {}

    for idx, trans in enumerate(transactions):
        seen = set()
        for item in trans:
            item_norm = item.strip().lower()
            if not item_norm or item_norm in seen:
                continue
            seen.add(item_norm)
            k = (item_norm,)
            counts.setdefault(k, {"count": 0, "rows": []})
            counts[k]["count"] += 1
            counts[k]["rows"].append(row_labels[idx])

    def prune(d):
        return {k: v for k, v in d.items() if v["count"] >= min_support}

    freq = [prune(counts)]
    if not freq[0]:
        return []

    k = 2
    while k <= max_size and freq[-1]:
        prev_keys = sorted(freq[-1].keys())
        candidates = set()
        for i in range(len(prev_keys)):
            for j in range(i+1, len(prev_keys)):
                a, b = prev_keys[i], prev_keys[j]
                if a[:k-2] == b[:k-2]:
                    merged = tuple(sorted(set(a) | set(b)))
                    if len(merged) == k:
                        if all(tuple(sorted(sub)) in freq[-1]
                               for sub in itertools.combinations(merged, k-1)):
                            candidates.add(merged)
        if not candidates:
            break
        cand_counts = {c: {"count": 0, "rows": []} for c in candidates}
        for idx, trans in enumerate(transactions):
            tset = {x.strip().lower() for x in trans}
            for c in candidates:
                if set(c).issubset(tset):
                    cand_counts[c]["count"] += 1
                    cand_counts[c]["rows"].append(row_labels[idx])
        freq.append(prune(cand_counts))
        k += 1

    out = []
    for level in freq:
        for itemset, info in level.items():
            out.append({"itemset": list(itemset), "count": info["count"], "rows": info["rows"]})
    return out

print("CSV loader + Apriori ready")


In [ ]:
# ── Cell 3: Model loading ──────────────────────────────────────────────────

_MODEL = None
_TOKENIZER = None

def load_model(model_path: str, hf_token: str = ""):
    global _MODEL, _TOKENIZER
    if _MODEL is not None:
        return _MODEL, _TOKENIZER

    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    print(f"Loading model: {model_path}")
    token_kwargs = {"token": hf_token} if hf_token else {}

    _TOKENIZER = AutoTokenizer.from_pretrained(model_path, **token_kwargs)

    if torch.cuda.is_available():
        _MODEL = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.float16, device_map="auto", **token_kwargs
        )
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
    else:
        _MODEL = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.float32, device_map="cpu", **token_kwargs
        )
        print("   CPU (will be slow)")

    _MODEL.eval()
    # Clear generation_config.max_length to avoid conflict with max_new_tokens
    if hasattr(_MODEL, 'generation_config') and getattr(_MODEL.generation_config, 'max_length', None) is not None:
        _MODEL.generation_config.max_length = None
    return _MODEL, _TOKENIZER

load_model(MODEL_REPO, HF_TOKEN)


In [ ]:
# ── Cell 4: Inference helpers ──────────────────────────────────────────────

def run_inference(csv_text, min_support, max_new_tokens=2048, temperature=0.3, two_phase=False):
    import torch
    model, tokenizer = _MODEL, _TOKENIZER

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Find all frequent itemsets with minimum support count = {min_support} "
            f"in this dataset:\n\n{csv_text}"
        )},
    ]

    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    if hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids
        attention_mask = getattr(encoded, "attention_mask", None)
    else:
        input_ids = encoded
        attention_mask = None

    if torch.cuda.is_available():
        input_ids = input_ids.to(model.device)
        if attention_mask is not None:
            attention_mask = attention_mask.to(model.device)

    gen_kwargs = dict(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True, top_k=50, top_p=0.90,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if attention_mask is not None:
        gen_kwargs["attention_mask"] = attention_mask

    with torch.no_grad():
        out = model.generate(**gen_kwargs)
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)


def extract_json(raw):
    has_think = "</think>" in raw
    json_text = raw
    if has_think:
        parts = raw.split("</think>", 1)
        json_text = parts[1].strip() if len(parts) > 1 else ""

    for text_source in [json_text, raw]:
        try:
            parsed = json.loads(text_source)
            if isinstance(parsed, list):
                return parsed, True
        except json.JSONDecodeError:
            pass
        m = re.search(r"\[.*\]", text_source, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group())
                if isinstance(parsed, list):
                    return parsed, True
            except json.JSONDecodeError:
                pass
    return [], False


def normalize_itemsets(raw_items):
    items = []
    for rec in raw_items:
        if not isinstance(rec, dict):
            continue
        itemset = rec.get("itemset", [])
        if not isinstance(itemset, list) or not itemset:
            continue
        count = rec.get("count", 0)
        rows = rec.get("rows", [])
        norm_set = sorted(str(x).strip().lower() for x in itemset)
        norm_rows = []
        for r in rows:
            if isinstance(r, int):
                norm_rows.append(f"Row {r}")
            elif isinstance(r, str) and re.match(r"Row \d+", r):
                norm_rows.append(r)
            else:
                try:
                    norm_rows.append(f"Row {int(r)}")
                except (ValueError, TypeError):
                    pass
        items.append({
            "itemset": norm_set,
            "count": count if isinstance(count, int) else 0,
            "rows": sorted(set(norm_rows)),
        })
    return items

print("Inference helpers ready")


In [ ]:
# ── Sanity: Quick single-inference test ─────────────────────────────────────
import torch, time

print("Running quick sanity test on one dataset...")
test_csv = "Row 1: bread, milk, eggs\nRow 2: bread, butter\nRow 3: milk, eggs, butter\nRow 4: bread, milk, eggs\nRow 5: bread, butter"

# No token budget limit for sanity - let it finish
t0 = time.perf_counter()
raw = run_inference(test_csv, min_support=3, max_new_tokens=256, temperature=0.3)
elapsed = time.perf_counter() - t0

parsed, ok = extract_json(raw)
items = normalize_itemsets(parsed)

print(f"Generated {len(raw)} chars in {elapsed:.1f}s")
print(f"Parse: {'OK' if ok else 'FAILED'}  |  Itemsets: {len(items)}")
if ok and items:
    for it in items[:3]:
        print(f"  {it['itemset']}: count={it['count']}")
print()
print("Sanity passed — proceeding with full eval")


In [ ]:
# ── Cell 6: Metrics + training exclusion ───────────────────────────────────

def canon(itemset):
    return frozenset(str(x).strip().lower() for x in itemset)

def compute_metrics(apriori_sets, model_sets, all_csv_items):
    apr_c = {canon(s["itemset"]) for s in apriori_sets}
    mod_c = {canon(s["itemset"]) for s in model_sets}
    tp = apr_c & mod_c
    fp = mod_c - apr_c
    fn_set = apr_c - mod_c

    precision = len(tp) / len(mod_c) if mod_c else 0.0
    recall = len(tp) / len(apr_c) if apr_c else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    apr_map = {canon(s["itemset"]): s["count"] for s in apriori_sets}
    count_ok = count_total = 0
    for s in model_sets:
        c_key = canon(s["itemset"])
        if c_key in apr_map:
            count_total += 1
            if abs(s["count"] - apr_map[c_key]) <= 1:
                count_ok += 1
    count_acc = count_ok / count_total if count_total else 0.0

    hal = sum(1 for s in model_sets
              if any(item not in all_csv_items for item in canon(s["itemset"])))
    hal_rate = hal / len(model_sets) if model_sets else 0.0

    return {
        "apriori_count": len(apr_c), "model_count": len(mod_c),
        "tp": len(tp), "fp": len(fp), "fn": len(fn_set),
        "precision": round(precision, 4), "recall": round(recall, 4),
        "f1": round(f1, 4), "exact_match": f1 == 1.0,
        "count_accuracy": round(count_acc, 4), "hallucination_rate": round(hal_rate, 4),
    }


def get_training_datasets():
    files = set()
    for base in [SCRIPT_DIR / "data", SCRIPT_DIR.parent / "data", Path.cwd() / "data"]:
        for json_name in ["sft_cot_v3.json", "dpo_real_v2.json"]:
            p = base / json_name
            if not p.exists():
                continue
            try:
                data = json.loads(p.read_text())
                for entry in data:
                    ds_id = entry.get("dataset_id", "")
                    fn = ds_id.split(":")[0] if ":" in ds_id else ds_id
                    if fn:
                        files.add(fn)
            except Exception:
                pass
    return files

print("Metrics + exclusion ready")


In [ ]:
# ── Cell 7: Select evaluation datasets ──────────────────────────────────────

# Find datasets directory
data_dir = Path.home() / DATA_DIR
if not data_dir.exists():
    data_dir = SCRIPT_DIR / DATA_DIR
if not data_dir.exists():
    data_dir = Path.cwd() / DATA_DIR
if not data_dir.exists():
    raise FileNotFoundError(f"Data dir not found: {DATA_DIR}")

all_csvs = sorted(data_dir.glob("*.csv"))
print(f"Found {len(all_csvs)} CSV datasets in {data_dir}")

if EXCLUDE_TRAIN:
    training = get_training_datasets()
    before = len(all_csvs)
    all_csvs = [f for f in all_csvs if f.name not in training]
    print(f"Excluded {before - len(all_csvs)} training ({len(all_csvs)} remaining)")

random.seed(SEED)
selected = random.sample(all_csvs, min(EVAL_COUNT, len(all_csvs)))
print(f"Selected {len(selected)} datasets")
print()


In [ ]:
# ── Cell 8: Run evaluation ──────────────────────────────────────────────────
import pandas as pd
import traceback

out_dir = SCRIPT_DIR / OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

results = []
total_start = time.perf_counter()

for i, csv_path in enumerate(selected, 1):
    dname = csv_path.name
    print(f"[{i}/{len(selected)}] {dname}")

    try:
        transactions, csv_text, n_rows, n_cols = load_csv(str(csv_path))

        all_items = set()
        for t in transactions:
            for item in t:
                all_items.add(item.strip().lower())

        apr_sets = apriori(transactions, MIN_SUPPORT, MAX_SIZE)
        print(f"   {n_rows}x{n_cols}  Apriori:{len(apr_sets):>4}  |  ", end="", flush=True)

        t1 = time.perf_counter()
        raw = run_inference(csv_text, MIN_SUPPORT, MAX_NEW_TOKENS, TEMPERATURE, TWO_PHASE)
        model_time = time.perf_counter() - t1

        parsed, parse_ok = extract_json(raw)
        model_sets = normalize_itemsets(parsed)
        print(f"Model:{len(model_sets):>4} ({model_time:.1f}s)  |  ", end="")

        m = compute_metrics(apr_sets, model_sets, all_items)
        print(f"F1={m['f1']:.1%} {'✅' if parse_ok else '❌parse'}")

        results.append(dict(
            dataset=dname, n_rows=n_rows, n_cols=n_cols,
            apriori_count=len(apr_sets), model_count=len(model_sets),
            model_time_s=round(model_time, 2), parse_ok=parse_ok, metrics=m))

        (out_dir / f"raw_{dname.replace('.csv','')}.txt").write_text(raw, encoding="utf-8")

    except Exception as exc:
        print(f"FAILED: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        results.append(dict(dataset=dname, error=str(exc) or type(exc).__name__))

total_time = time.perf_counter() - total_start
print(f"\nTotal: {total_time:.0f}s")


In [ ]:
# ── Cell 9: Aggregate summary ───────────────────────────────────────────────

ok = [r for r in results if "metrics" in r]
if not ok:
    print("No successful evaluations")
else:
    n = len(ok)
    summary = {}
    summary["model"] = MODEL_REPO
    summary["datasets_evaluated"] = n
    summary["datasets_attempted"] = len(selected)
    summary["avg_precision"]    = round(sum(r["metrics"]["precision"] for r in ok) / n, 4)
    summary["avg_recall"]       = round(sum(r["metrics"]["recall"] for r in ok) / n, 4)
    summary["avg_f1"]           = round(sum(r["metrics"]["f1"] for r in ok) / n, 4)
    exact_n = sum(1 for r in ok if r["metrics"]["exact_match"])
    summary["exact_matches"]    = exact_n
    summary["exact_match_rate"] = round(exact_n / n, 4)
    summary["parse_rate"]       = round(sum(1 for r in ok if r["parse_ok"]) / n, 4)
    summary["avg_hallucination"]= round(sum(r["metrics"]["hallucination_rate"] for r in ok) / n, 4)
    summary["avg_count_accuracy"] = round(sum(r["metrics"]["count_accuracy"] for r in ok) / n, 4)
    summary["avg_time_s"]       = round(sum(r["model_time_s"] for r in ok) / n, 2)
    summary["total_time_s"]     = round(total_time, 1)

    print("=" * 60)
    print("  VERIFICATION SUMMARY")
    print("=" * 60)
    print(f"  Datasets:      {n}")
    print(f"  Avg F1:        {summary['avg_f1']:.2%}")
    print(f"  Avg Precision: {summary['avg_precision']:.2%}")
    print(f"  Avg Recall:    {summary['avg_recall']:.2%}")
    print(f"  Exact Match:   {exact_n}/{n} ({summary['exact_match_rate']:.1%})")
    print(f"  Parse Rate:    {summary['parse_rate']:.1%}")
    print(f"  Hallucination: {summary['avg_hallucination']:.2%}")
    print(f"  Count Acc:     {summary['avg_count_accuracy']:.2%}")
    print(f"  Avg Time:      {summary['avg_time_s']:.1f}s/dataset")
    print(f"  Total Time:    {summary['total_time_s']:.0f}s")


In [ ]:
# ── Cell 10: Per-dataset table ───────────────────────────────────────────────

if ok:
    print()
    print("-" * 85)
    hdr = f"{'Dataset':<40} {'Size':>6} {'Apriori':>7} {'Model':>7} {'TP':>4} {'FP':>4} {'FN':>4} {'P%':>5} {'R%':>5} {'F1%':>5}"
    print(hdr)
    print("-" * 85)
    for r in ok:
        m = r["metrics"]
        size = f"{r['n_rows']}x{r['n_cols']}"
        print(f"{r['dataset']:<40} {size:>6} "
              f"{r['apriori_count']:>7} {r['model_count']:>7} "
              f"{m['tp']:>4} {m['fp']:>4} {m['fn']:>4} "
              f"{int(m['precision']*100):>5} {int(m['recall']*100):>5} {int(m['f1']*100):>5}")


In [ ]:
# ── Cell 11: Compare to expected result ─────────────────────────────────────

if ok:
    avg = summary["avg_f1"]
    print()
    print("=" * 60)
    print("  COMPARISON TO EXPECTED RESULT")
    print("=" * 60)
    print(f"  Expected:   avg F1 ~ 0.13  (Iteration 4 SFT, primary_v3)")
    print(f"  Observed:   avg F1 ~ {avg:.2f}")
    if 0.10 <= avg <= 0.16:
        print("  \u2705 F1 in expected range \u2014 this IS the correct SFT model")
    elif avg > 0.16:
        print(f"  \u26a0\ufe0f  F1 ({avg:.2f}) HIGHER than expected (0.13) \u2014 may be DPO?")
    else:
        print(f"  \u26a0\ufe0f  F1 ({avg:.2f}) LOWER than expected (0.13)")
    print("=" * 60)


In [ ]:
# ── Cell 12: Save results ───────────────────────────────────────────────────

if ok:
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
    (out_dir / "per_dataset.json").write_text(json.dumps(ok, indent=2, ensure_ascii=False))
    print(f"\nResults saved to {out_dir}/")
    print(f"  summary.json      \u2014 aggregate metrics")
    print(f"  per_dataset.json  \u2014 per-dataset breakdown")
    print(f"  raw_*.txt         \u2014 raw model outputs")
